In [1]:
!pip -q install xgboost gradio holidays

In [2]:
import numpy as np
import pandas as pd
import holidays
import gradio as gr
import matplotlib.pyplot as plt

from datetime import datetime, timedelta
from google.colab import drive
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
base_path = "/content/drive/MyDrive/smart-city-traffic-patterns"
train_path = f"{base_path}/train_aWnotuB.csv"

train_df = pd.read_csv(train_path)
train_df["DateTime"] = pd.to_datetime(train_df["DateTime"])
train_df = train_df.sort_values(["Junction", "DateTime"]).reset_index(drop=True)

In [5]:
india_holidays = holidays.India()

def create_features(df):
    df = df.copy()
    df["year"] = df["DateTime"].dt.year
    df["month"] = df["DateTime"].dt.month
    df["day"] = df["DateTime"].dt.day
    df["hour"] = df["DateTime"].dt.hour
    df["dayofweek"] = df["DateTime"].dt.dayofweek
    df["weekofyear"] = df["DateTime"].dt.isocalendar().week.astype(int)
    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
    df["is_holiday"] = df["DateTime"].dt.date.isin(india_holidays).astype(int)
    df["morning_peak"] = df["hour"].between(7, 10).astype(int)
    df["evening_peak"] = df["hour"].between(16, 19).astype(int)
    df["night"] = df["hour"].between(0, 5).astype(int)
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["dow_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)
    return df

train_feat = create_features(train_df)

In [6]:
for lag in [1, 2, 3, 24, 168]:
    train_feat[f"lag_{lag}"] = train_feat.groupby("Junction")["Vehicles"].shift(lag)

for window in [3, 6, 24]:
    train_feat[f"rolling_mean_{window}"] = (
        train_feat.groupby("Junction")["Vehicles"]
        .shift(1)
        .rolling(window=window)
        .mean()
        .reset_index(level=0, drop=True)
    )

train_feat = train_feat.dropna().reset_index(drop=True)

feature_cols = [
    "Junction", "year", "month", "day", "hour", "dayofweek", "weekofyear",
    "is_weekend", "is_holiday", "morning_peak", "evening_peak", "night",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos",
    "lag_1", "lag_2", "lag_3", "lag_24", "lag_168",
    "rolling_mean_3", "rolling_mean_6", "rolling_mean_24"
]

X = train_feat[feature_cols]
y = train_feat["Vehicles"]

In [7]:
split_idx = int(len(train_feat) * 0.8)
X_train, X_val = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_val = y.iloc[:split_idx], y.iloc[split_idx:]

model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.85,
    colsample_bytree=0.85,
    objective="reg:squarederror",
    random_state=42
)
model.fit(X_train, y_train)

val_pred = np.maximum(model.predict(X_val), 0)
mae = mean_absolute_error(y_val, val_pred)
rmse = np.sqrt(mean_squared_error(y_val, val_pred))
r2 = r2_score(y_val, val_pred)

history_df = train_df.copy()
junction_values = sorted(history_df["Junction"].unique().tolist())

In [8]:
def traffic_label(pred):
    if pred < 100:
        return "Low"
    elif pred < 300:
        return "Medium"
    elif pred < 500:
        return "High"
    return "Severe"

def traffic_color(pred):
    if pred < 100:
        return "#15803d"
    elif pred < 300:
        return "#ca8a04"
    elif pred < 500:
        return "#ea580c"
    return "#dc2626"

def build_feature_row(junction, dt_obj, holiday_override):
    junction_df = history_df[history_df["Junction"] == junction].sort_values("DateTime")
    past = junction_df[junction_df["DateTime"] < dt_obj]

    if len(past) == 0:
        return None

    dayofweek = dt_obj.weekday()
    hour = dt_obj.hour

    row = pd.DataFrame([{
        "Junction": junction,
        "year": dt_obj.year,
        "month": dt_obj.month,
        "day": dt_obj.day,
        "hour": hour,
        "dayofweek": dayofweek,
        "weekofyear": int(dt_obj.isocalendar().week),
        "is_weekend": int(dayofweek >= 5),
        "is_holiday": int(holiday_override),
        "morning_peak": int(7 <= hour <= 10),
        "evening_peak": int(16 <= hour <= 19),
        "night": int(0 <= hour <= 5),
        "hour_sin": np.sin(2 * np.pi * hour / 24),
        "hour_cos": np.cos(2 * np.pi * hour / 24),
        "dow_sin": np.sin(2 * np.pi * dayofweek / 7),
        "dow_cos": np.cos(2 * np.pi * dayofweek / 7),
        "lag_1": past["Vehicles"].iloc[-1] if len(past) >= 1 else past["Vehicles"].median(),
        "lag_2": past["Vehicles"].iloc[-2] if len(past) >= 2 else past["Vehicles"].median(),
        "lag_3": past["Vehicles"].iloc[-3] if len(past) >= 3 else past["Vehicles"].median(),
        "lag_24": past["Vehicles"].iloc[-24] if len(past) >= 24 else past["Vehicles"].median(),
        "lag_168": past["Vehicles"].iloc[-168] if len(past) >= 168 else past["Vehicles"].median(),
        "rolling_mean_3": past["Vehicles"].tail(3).mean(),
        "rolling_mean_6": past["Vehicles"].tail(6).mean(),
        "rolling_mean_24": past["Vehicles"].tail(24).mean()
    }])

    return row[feature_cols]

In [9]:
def make_plot(junction):
    temp = history_df[history_df["Junction"] == junction].sort_values("DateTime").tail(200)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(temp["DateTime"], temp["Vehicles"], color="teal", linewidth=2)
    ax.set_title(f"Traffic History for Junction {junction}")
    ax.set_xlabel("DateTime")
    ax.set_ylabel("Vehicles")
    plt.xticks(rotation=45)
    plt.tight_layout()
    return fig